### Imports + paths + connect to DuckDB

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

import duckdb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("CWD:", Path.cwd())

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists():
            return p
        # fallback if you don't have .git in your working copy
        if (p / "Day-11").exists() or (p / "day-11").exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
print("REPO_ROOT:", REPO_ROOT)

# Find the duckdb file anywhere under the repo
duckdb_files = list(REPO_ROOT.glob("**/*.duckdb"))
print("Found duckdb files:", [str(p.relative_to(REPO_ROOT)) for p in duckdb_files])

# Prefer day11_noshow.duckdb if it exists
DB_PATH = None
for p in duckdb_files:
    if "day11_noshow.duckdb" in p.name.lower():
        DB_PATH = p
        break

if DB_PATH is None and duckdb_files:
    DB_PATH = duckdb_files[0]

if DB_PATH is None:
    raise FileNotFoundError(
        "No .duckdb file found anywhere under REPO_ROOT.\n"
        "Confirm Day-11/data/warehouse/day11_noshow.duckdb exists."
    )

print("Using DB_PATH:", DB_PATH)

DAY17_DIR = REPO_ROOT / "Day-17"
REPORTS = DAY17_DIR / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH), read_only=True)
print("Tables:", con.execute("SHOW TABLES").fetchall())


CWD: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-17\notebooks
REPO_ROOT: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science
Found duckdb files: ['Day-1\\data\\warehouse\\day1.duckdb', 'Day-11\\data\\warehouse\\day11_noshow.duckdb']
Using DB_PATH: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-11\data\warehouse\day11_noshow.duckdb
Tables: [('bronze_appointments',), ('gold_appointments_base',), ('gold_appointments_features_v1',), ('gold_appointments_features_v1_patient_split',), ('silver_appointments',), ('split_patient_v1',)]


### Pick the gold/silver table automatically

In [3]:
tables = [t[0] for t in con.execute("SHOW TABLES").fetchall()]
tables_lower = {t: t.lower() for t in tables}

def pick_table(candidates=("gold", "features", "silver", "clean", "appointments")):
    scored = []
    for t in tables:
        score = sum(1 for k in candidates if k in tables_lower[t])
        if score > 0:
            scored.append((score, t))
    scored.sort(reverse=True)
    return scored[0][1] if scored else None

TABLE = pick_table()

if TABLE is None:
    raise ValueError(
        "Could not auto-pick a table. Set TABLE manually from the printed list above."
    )

print("Using TABLE:", TABLE)


Using TABLE: gold_appointments_features_v1_patient_split


### Load modeling frame (A, Y, X) from DuckDB

In [4]:
A_COL = "sms_received"
Y_COL = "label"

# Candidate covariates (edit freely; we’ll keep only those that exist in the table)
candidate_covars = [
    "age","gender","neighbourhood",
    "scholarship","hipertension","diabetes","alcoholism","handcap",
    "lead_time_days","lead_time_clipped","lead_time_log1p","lead_time_bin",
    "appt_dow","appt_month","sched_dow","sched_month","sched_hour",
    "nbhd_n","prior_appt_count"
]

# Pull schema so we only select columns that exist
schema_df = con.execute(f"DESCRIBE {TABLE}").df()
cols = schema_df["column_name"].tolist()

use_covars = [c for c in candidate_covars if c in cols]
needed = ["appointment_id", "person_id", A_COL, Y_COL]  # IDs optional but useful
select_cols = [c for c in needed if c in cols] + use_covars

q = f"SELECT {', '.join(select_cols)} FROM {TABLE}"
df = con.execute(q).df()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Clean A and Y to {0,1}
df = df.dropna(subset=[A_COL, Y_COL]).copy()
df[A_COL] = df[A_COL].astype(int)
df[Y_COL] = df[Y_COL].astype(int)

print("\nA value counts:\n", df[A_COL].value_counts())
print("\nY value counts:\n", df[Y_COL].value_counts())
print("\nNaive mean(Y) by A:\n", df.groupby(A_COL)[Y_COL].mean().to_frame("mean_Y").assign(n=df.groupby(A_COL).size()))


Shape: (110516, 23)
Columns: ['appointment_id', 'person_id', 'sms_received', 'label', 'age', 'gender', 'neighbourhood', 'scholarship', 'hipertension', 'diabetes', 'alcoholism', 'handcap', 'lead_time_days', 'lead_time_clipped', 'lead_time_log1p', 'lead_time_bin', 'appt_dow', 'appt_month', 'sched_dow', 'sched_month', 'sched_hour', 'nbhd_n', 'prior_appt_count']

A value counts:
 sms_received
0    75035
1    35481
Name: count, dtype: int64

Y value counts:
 label
0    88205
1    22311
Name: count, dtype: int64

Naive mean(Y) by A:
                 mean_Y      n
sms_received                 
0             0.166949  75035
1             0.275753  35481


### Build the propensity model (Day 16 but reproducible here)

In [5]:
X_cols = use_covars
X = df[X_cols].copy()
A = df[A_COL].to_numpy()

# Decide types
categorical_cols = [c for c in X_cols if str(X[c].dtype) == "object"]
numeric_cols = [c for c in X_cols if c not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ],
    remainder="drop"
)

prop_model = LogisticRegression(max_iter=3000, solver="lbfgs")
prop_pipe = Pipeline(steps=[("prep", preprocess), ("clf", prop_model)])

X_tr, X_te, A_tr, A_te = train_test_split(X, A, test_size=0.25, random_state=42, stratify=A)
prop_pipe.fit(X_tr, A_tr)

ps_te = prop_pipe.predict_proba(X_te)[:, 1]
print("Propensity ROC-AUC (held-out):", round(roc_auc_score(A_te, ps_te), 4))

df["pscore"] = prop_pipe.predict_proba(X)[:, 1]
df[["pscore", A_COL, Y_COL]].head()


C:\Users\sarfo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Propensity ROC-AUC (held-out): 0.8919


,pscore,sms_received,label
0,0.001938,0,0
1,0.001428,0,0
2,0.001398,0,0
3,0.690386,0,1
4,0.683785,0,1


### Overlap / positivity check + save plot

In [6]:
def summarize_ps(df, a_col="sms_received", ps_col="pscore"):
    out = []
    for a in [0,1]:
        s = df.loc[df[a_col]==a, ps_col]
        out.append({
            "A": a,
            "n": int(s.shape[0]),
            "min": float(s.min()),
            "p01": float(s.quantile(0.01)),
            "p05": float(s.quantile(0.05)),
            "median": float(s.median()),
            "p95": float(s.quantile(0.95)),
            "p99": float(s.quantile(0.99)),
            "max": float(s.max()),
        })
    return pd.DataFrame(out)

ps_summary = summarize_ps(df, A_COL, "pscore")
ps_summary.to_csv(REPORTS / "DAY17_pscore_summary.csv", index=False)
print(ps_summary)

plt.figure()
plt.hist(df.loc[df[A_COL]==0, "pscore"], bins=50, alpha=0.6, label="A=0")
plt.hist(df.loc[df[A_COL]==1, "pscore"], bins=50, alpha=0.6, label="A=1")
plt.xlabel("Propensity score P(A=1|X)")
plt.ylabel("Count")
plt.title("Overlap check: propensity score distributions")
plt.legend()
plt.savefig(REPORTS / "DAY17_overlap_hist.png", dpi=200, bbox_inches="tight")
plt.close()


   A      n       min       p01       p05    median       p95       p99  \
0  0  75035  0.000217  0.000596  0.001079  0.003315  0.662276  0.756475   
1  1  35481  0.008491  0.272753  0.392010  0.623831  0.803067  0.839331   

        max  
0  0.873613  
1  0.889876  


### IPW (Hájek) estimator + bootstrap CI

In [7]:
Y = df[Y_COL].to_numpy()
ps = df["pscore"].to_numpy()

# optional trimming to improve positivity
TRIM = (0.01, 0.99)  # change to (0.05,0.95) if overlap is bad
mask = (ps >= TRIM[0]) & (ps <= TRIM[1])
df_ipw = df.loc[mask].copy()

A = df_ipw[A_COL].to_numpy()
Y = df_ipw[Y_COL].to_numpy()
ps = df_ipw["pscore"].to_numpy()

pA = A.mean()
sw = A * (pA / ps) + (1 - A) * ((1 - pA) / (1 - ps))  # stabilized

def hajek_mean(y, w):
    return (w * y).sum() / w.sum()

# IPW estimates of E[Y(1)] and E[Y(0)] (stabilized + normalized within groups)
w1 = sw * A
w0 = sw * (1 - A)

EY1 = hajek_mean(Y[A==1], w1[A==1])
EY0 = hajek_mean(Y[A==0], w0[A==0])
ATE_ipw = EY1 - EY0

def ess(w):
    return (w.sum()**2) / (np.square(w).sum())

ipw_stats = {
    "trim_lo": TRIM[0],
    "trim_hi": TRIM[1],
    "n_used": int(len(df_ipw)),
    "pA": float(pA),
    "EY1_ipw": float(EY1),
    "EY0_ipw": float(EY0),
    "ATE_ipw": float(ATE_ipw),
    "ESS_treated": float(ess(w1[A==1])),
    "ESS_control": float(ess(w0[A==0])),
}
print(ipw_stats)

# Bootstrap CI (nonparametric)
def ipw_ate_bootstrap(df_in, B=200, seed=7):
    rng = np.random.default_rng(seed)
    n = len(df_in)
    ates = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        d = df_in.iloc[idx]
        A = d[A_COL].to_numpy()
        Y = d[Y_COL].to_numpy()
        ps = d["pscore"].to_numpy()
        pA = A.mean()
        sw = A * (pA / ps) + (1 - A) * ((1 - pA) / (1 - ps))
        w1 = sw * A
        w0 = sw * (1 - A)
        EY1 = (w1[A==1] * Y[A==1]).sum() / w1[A==1].sum()
        EY0 = (w0[A==0] * Y[A==0]).sum() / w0[A==0].sum()
        ates.append(EY1 - EY0)
    ates = np.array(ates)
    return float(np.mean(ates)), float(np.quantile(ates, 0.025)), float(np.quantile(ates, 0.975))

ipw_mean, ipw_lo, ipw_hi = ipw_ate_bootstrap(df_ipw, B=200)
print("IPW ATE (bootstrap mean, 95% CI):", ipw_mean, (ipw_lo, ipw_hi))


{'trim_lo': 0.01, 'trim_hi': 0.99, 'n_used': 61390, 'pA': 0.5779117120052126, 'EY1_ipw': 0.27900451403527216, 'EY0_ipw': 0.3250591769591066, 'ATE_ipw': -0.04605466292383442, 'ESS_treated': 15311.02687696283, 'ESS_control': 23436.855954087572}
IPW ATE (bootstrap mean, 95% CI): -0.04617820311445211 (-0.056290428001043114, -0.037508429790076674)


### Build a reusable (unfitted) preprocess

In [9]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler

# Build an *unfitted* base preprocess. We'll clone it per model.
base_out_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())   # helps convergence
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ],
    remainder="drop"
)


### Fit μ1 and μ0 with separate preprocess objects (no sharing)

In [10]:
# separate preprocess objects (critical)
prep1 = clone(base_out_preprocess)
prep0 = clone(base_out_preprocess)

pipe_y1 = Pipeline(steps=[
    ("prep", prep1),
    ("clf", LogisticRegression(max_iter=6000, solver="saga"))  # saga handles sparse well
])

pipe_y0 = Pipeline(steps=[
    ("prep", prep0),
    ("clf", LogisticRegression(max_iter=6000, solver="saga"))
])

A = df_ipw[A_COL].astype(int).to_numpy()
Y = df_ipw[Y_COL].astype(int).to_numpy()
ps = df_ipw["pscore"].astype(float).to_numpy()

# clip ps for stability (positivity protection)
eps = 1e-3
ps = np.clip(ps, eps, 1 - eps)

X = df_ipw[X_cols].copy()

pipe_y1.fit(X[A==1], Y[A==1])
pipe_y0.fit(X[A==0], Y[A==0])

mu1 = pipe_y1.predict_proba(X)[:, 1]
mu0 = pipe_y0.predict_proba(X)[:, 1]


### AIPW ATE + fast (asymptotic) 95% CI (no bootstrap needed)

In [11]:
aipw_terms = (mu1 - mu0) + (A * (Y - mu1) / ps) - ((1 - A) * (Y - mu0) / (1 - ps))
ATE_aipw = float(np.mean(aipw_terms))

# Influence-function SE
n = len(aipw_terms)
se = float(np.std(aipw_terms - ATE_aipw, ddof=1) / np.sqrt(n))
ci = (ATE_aipw - 1.96*se, ATE_aipw + 1.96*se)

print("AIPW ATE:", ATE_aipw)
print("AIPW SE:", se)
print("AIPW 95% CI:", ci)


AIPW ATE: -0.04229419911236055
AIPW SE: 0.004989005149814082
AIPW 95% CI: (-0.05207264920599615, -0.03251574901872495)


### Bootstrap (optional) with correct cloning (won’t crash)

In [12]:
def aipw_ate_bootstrap(df_in, B=100, seed=11):
    rng = np.random.default_rng(seed)
    n = len(df_in)
    ates = []

    for _ in range(B):
        idx = rng.integers(0, n, n)
        d = df_in.iloc[idx].copy()

        Xb = d[X_cols].copy()
        Ab = d[A_COL].astype(int).to_numpy()
        Yb = d[Y_COL].astype(int).to_numpy()
        psb = np.clip(d["pscore"].astype(float).to_numpy(), 1e-3, 1-1e-3)

        # IMPORTANT: clone preprocess separately for each pipe
        pipe1 = Pipeline(steps=[
            ("prep", clone(base_out_preprocess)),
            ("clf", LogisticRegression(max_iter=6000, solver="saga"))
        ])
        pipe0 = Pipeline(steps=[
            ("prep", clone(base_out_preprocess)),
            ("clf", LogisticRegression(max_iter=6000, solver="saga"))
        ])

        pipe1.fit(Xb[Ab==1], Yb[Ab==1])
        pipe0.fit(Xb[Ab==0], Yb[Ab==0])

        mu1b = pipe1.predict_proba(Xb)[:, 1]
        mu0b = pipe0.predict_proba(Xb)[:, 1]

        term = (mu1b - mu0b) + (Ab * (Yb - mu1b) / psb) - ((1 - Ab) * (Yb - mu0b) / (1 - psb))
        ates.append(np.mean(term))

    ates = np.array(ates)
    return float(np.mean(ates)), float(np.quantile(ates, 0.025)), float(np.quantile(ates, 0.975))

aipw_mean, aipw_lo, aipw_hi = aipw_ate_bootstrap(df_ipw, B=100)
print("AIPW bootstrap mean, 95% CI:", aipw_mean, (aipw_lo, aipw_hi))


AIPW bootstrap mean, 95% CI: -0.04225075261649083 (-0.05149046167603051, -0.03375771412522062)


### Save Day 17 results (single CSV + JSON)

In [13]:
naive = df.groupby(A_COL)[Y_COL].mean()
naive_ate = float(naive.loc[1] - naive.loc[0])

# If you used asymptotic AIPW CI earlier:
# aipw_lo, aipw_hi = ci

results = pd.DataFrame([{
    "estimand": "ATE of SMS on Y (label)",
    "Y_definition_note": "Verify: label=1 likely means no-show; interpret sign accordingly",
    "trim": f"{TRIM[0]}-{TRIM[1]}",
    "naive_diff_in_means": naive_ate,
    "ipw_ate": float(ATE_ipw),
    "ipw_ci_low": float(ipw_lo),
    "ipw_ci_high": float(ipw_hi),
    "aipw_ate": float(ATE_aipw),
    "aipw_ci_low": float(aipw_lo),
    "aipw_ci_high": float(aipw_hi),
    "n_used": int(len(df_ipw)),
}])

results.to_csv(REPORTS / "DAY17_ate_results.csv", index=False)

payload = {
    "naive": {"mean_Y_A0": float(naive.loc[0]), "mean_Y_A1": float(naive.loc[1]), "ATE_naive": naive_ate},
    "ipw": {**ipw_stats, "ci_low": float(ipw_lo), "ci_high": float(ipw_hi)},
    "aipw": {"ATE": float(ATE_aipw), "ci_low": float(aipw_lo), "ci_high": float(aipw_hi)},
}

with open(REPORTS / "DAY17_results.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Saved:", REPORTS)
print(results)


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-17\reports
                  estimand                                  Y_definition_note  \
0  ATE of SMS on Y (label)  Verify: label=1 likely means no-show; interpre...   

        trim  naive_diff_in_means   ipw_ate  ipw_ci_low  ipw_ci_high  \
0  0.01-0.99             0.108804 -0.046055    -0.05629    -0.037508   

   aipw_ate  aipw_ci_low  aipw_ci_high  n_used  
0 -0.042294     -0.05149     -0.033758   61390  
